# 06 - Build VisionStudio Android App | Camera + Video

يبني هذا الدفتر APK إصدار 0.2.0 بسرعة للتسليم.

الميزات:
- اختيار صورة من المعرض.
- التقاط صورة بالكاميرا.
- اختيار فيديو واستخراج إطار أول للمعالجة.
- Grayscale / Threshold / Edges.
- حفظ النتيجة.

ملاحظة: معالجة فيديو كامل و YOLO الكامل أثقل ويحتاجان دفترًا تالياً للاختبار، لكن هذا يغطي مسار الكاميرا والفيديو الأساسي بسرعة.


> هذا الدفتر الموحّد يعتمد الإصدار 12 باعتباره النسخة الأحدث من مشروع Android، ويضم وظائف الإصدار 11 الأساسية ضمنه. لم يتم تشغيل كود الإصدار 11 القديم فوق الإصدار 12 حتى لا تُستبدل ملفات التطبيق الأحدث.


## 1. إعداد المسارات


In [1]:
from pathlib import Path

USE_DRIVE = True
PROJECT_NAME = 'VisionStudioAndroid'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = Path('/content/drive/MyDrive/VisionStudio_Android_Build')
else:
    WORK_DIR = Path('/content/VisionStudio_Android_Build')

APP_DIR = WORK_DIR / PROJECT_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('APP_DIR =', APP_DIR)


Mounted at /content/drive
APP_DIR = /content/drive/MyDrive/VisionStudio_Android_Build/VisionStudioAndroid


## فحص بيئة Java/Gradle

تم الاحتفاظ بهذه الخطوة المفيدة من الدفتر 11 قبل متابعة بناء النسخة النهائية.


In [3]:
!java -version
!which java || true
!gradle -version || true


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)
/usr/bin/java
/bin/bash: line 1: gradle: command not found


## 2. تثبيت Android SDK داخل Colab

قد تستغرق هذه الخلية من 2 إلى 6 دقائق أول مرة.


In [2]:
%%bash
set -e
export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT=/content/android-sdk
mkdir -p "$ANDROID_HOME/cmdline-tools"

if [ ! -x "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" ]; then
  cd /content
  rm -rf commandlinetools.zip cmdline-tools
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O commandlinetools.zip
  unzip -q commandlinetools.zip -d cmdline-tools
  rm -rf "$ANDROID_HOME/cmdline-tools/latest"
  mv cmdline-tools/cmdline-tools "$ANDROID_HOME/cmdline-tools/latest"
fi

yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --licenses >/dev/null
yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" "platform-tools" "platforms;android-35" "build-tools;35.0.0" >/dev/null

echo "Android SDK ready at $ANDROID_HOME"


Android SDK ready at /content/android-sdk


## 3. كتابة مشروع Android 0.2.0


In [3]:
from pathlib import Path
import textwrap

APP_DIR.mkdir(parents=True, exist_ok=True)

def write_file(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding='utf-8')

write_file(APP_DIR / 'settings.gradle', '''
pluginManagement {
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}

dependencyResolutionManagement {
    repositoriesMode.set(RepositoriesMode.FAIL_ON_PROJECT_REPOS)
    repositories { google(); mavenCentral() }
}

rootProject.name = "VisionStudio"
include ":app"
''')

write_file(APP_DIR / 'build.gradle', '''
plugins { id "com.android.application" version "8.7.3" apply false }
''')

write_file(APP_DIR / 'local.properties', '''
sdk.dir=/content/android-sdk
''')

write_file(APP_DIR / 'app/build.gradle', '''
plugins { id "com.android.application" }

android {
    namespace "com.visionstudio.app"
    compileSdk 35
    defaultConfig {
        applicationId "com.visionstudio.app"
        minSdk 26
        targetSdk 35
        versionCode 2
        versionName "0.2.0"
    }
}
''')

write_file(APP_DIR / 'app/src/main/res/values/styles.xml', '''
<?xml version="1.0" encoding="utf-8"?>
<resources>
    <style name="AppTheme" parent="android:style/Theme.Material.Light.NoActionBar">
        <item name="android:fontFamily">sans</item>
        <item name="android:windowLightStatusBar">true</item>
        <item name="android:navigationBarColor">#F7F8FA</item>
        <item name="android:colorAccent">#00796B</item>
    </style>
</resources>
''')

write_file(APP_DIR / 'app/src/main/AndroidManifest.xml', '''
<?xml version="1.0" encoding="utf-8"?>
<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-feature android:name="android.hardware.camera" android:required="false" />
    <application android:allowBackup="true" android:label="VisionStudio" android:theme="@style/AppTheme">
        <activity android:name=".MainActivity" android:exported="true">
            <intent-filter>
                <action android:name="android.intent.action.MAIN" />
                <category android:name="android.intent.category.LAUNCHER" />
            </intent-filter>
        </activity>
    </application>
</manifest>
''')

write_file(APP_DIR / 'app/src/main/java/com/visionstudio/app/MainActivity.java', r'''
package com.visionstudio.app;

import android.app.Activity;
import android.content.ContentValues;
import android.content.Intent;
import android.graphics.Bitmap;
import android.graphics.BitmapFactory;
import android.graphics.Color;
import android.media.MediaMetadataRetriever;
import android.net.Uri;
import android.os.Bundle;
import android.provider.MediaStore;
import android.view.Gravity;
import android.view.View;
import android.widget.Button;
import android.widget.ImageView;
import android.widget.LinearLayout;
import android.widget.ScrollView;
import android.widget.TextView;
import android.widget.Toast;

import java.io.IOException;
import java.io.InputStream;
import java.io.OutputStream;

public class MainActivity extends Activity {
    private static final int PICK_IMAGE_REQUEST = 1001;
    private static final int CAPTURE_IMAGE_REQUEST = 1002;
    private static final int PICK_VIDEO_REQUEST = 1003;
    private ImageView imageView;
    private TextView statusText;
    private Bitmap originalBitmap;
    private Bitmap currentBitmap;

    @Override public void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        setContentView(createLayout());
    }

    private View createLayout() {
        ScrollView scrollView = new ScrollView(this);
        LinearLayout root = new LinearLayout(this);
        root.setOrientation(LinearLayout.VERTICAL);
        root.setPadding(28, 32, 28, 32);
        root.setBackgroundColor(Color.rgb(247, 248, 250));
        scrollView.addView(root);
        TextView title = new TextView(this);
        title.setText("VisionStudio");
        title.setTextSize(26);
        title.setTextColor(Color.rgb(22, 29, 35));
        title.setGravity(Gravity.CENTER_HORIZONTAL);
        root.addView(title, fullWidth());
        statusText = new TextView(this);
        statusText.setText("اختر صورة، التقط بالكاميرا، أو اختر فيديو");
        statusText.setTextSize(15);
        statusText.setGravity(Gravity.CENTER_HORIZONTAL);
        statusText.setPadding(0, 12, 0, 18);
        root.addView(statusText, fullWidth());
        imageView = new ImageView(this);
        imageView.setAdjustViewBounds(true);
        imageView.setBackgroundColor(Color.WHITE);
        imageView.setScaleType(ImageView.ScaleType.FIT_CENTER);
        root.addView(imageView, new LinearLayout.LayoutParams(LinearLayout.LayoutParams.MATCH_PARENT, 900));
        LinearLayout buttons = new LinearLayout(this);
        buttons.setOrientation(LinearLayout.VERTICAL);
        buttons.setPadding(0, 22, 0, 0);
        root.addView(buttons, fullWidth());
        addButton(buttons, "اختيار صورة", v -> pickImage());
        addButton(buttons, "التقاط بالكاميرا", v -> captureImage());
        addButton(buttons, "اختيار فيديو واستخراج إطار", v -> pickVideo());
        addButton(buttons, "Grayscale", v -> applyFilter(FilterMode.GRAYSCALE));
        addButton(buttons, "Threshold", v -> applyFilter(FilterMode.THRESHOLD));
        addButton(buttons, "Edges", v -> applyFilter(FilterMode.EDGES));
        addButton(buttons, "استعادة الأصل", v -> restoreOriginal());
        addButton(buttons, "حفظ النتيجة", v -> saveCurrentImage());
        return scrollView;
    }

    private LinearLayout.LayoutParams fullWidth() { return new LinearLayout.LayoutParams(LinearLayout.LayoutParams.MATCH_PARENT, LinearLayout.LayoutParams.WRAP_CONTENT); }
    private void addButton(LinearLayout parent, String label, View.OnClickListener listener) {
        Button button = new Button(this);
        button.setText(label); button.setAllCaps(false); button.setTextSize(16); button.setOnClickListener(listener);
        LinearLayout.LayoutParams params = fullWidth(); params.setMargins(0, 8, 0, 8); parent.addView(button, params);
    }
    private void pickImage() { startActivityForResult(new Intent(Intent.ACTION_PICK, MediaStore.Images.Media.EXTERNAL_CONTENT_URI), PICK_IMAGE_REQUEST); }
    private void captureImage() {
        Intent intent = new Intent(MediaStore.ACTION_IMAGE_CAPTURE);
        if (intent.resolveActivity(getPackageManager()) == null) { showMessage("تطبيق الكاميرا غير متاح"); return; }
        startActivityForResult(intent, CAPTURE_IMAGE_REQUEST);
    }
    private void pickVideo() { startActivityForResult(new Intent(Intent.ACTION_PICK, MediaStore.Video.Media.EXTERNAL_CONTENT_URI), PICK_VIDEO_REQUEST); }

    @Override protected void onActivityResult(int requestCode, int resultCode, Intent data) {
        super.onActivityResult(requestCode, resultCode, data);
        if (resultCode != RESULT_OK || data == null) return;
        if (requestCode == PICK_IMAGE_REQUEST) loadImageUri(data.getData());
        else if (requestCode == CAPTURE_IMAGE_REQUEST) {
            Bundle extras = data.getExtras();
            Bitmap captured = extras == null ? null : (Bitmap) extras.get("data");
            if (captured == null) { showMessage("لم يتم التقاط صورة"); return; }
            setCurrentBitmap(captured, "تم التقاط صورة بالكاميرا");
        } else if (requestCode == PICK_VIDEO_REQUEST) loadVideoFrame(data.getData());
    }

    private void loadImageUri(Uri uri) {
        if (uri == null) { showMessage("لم يتم العثور على مسار الصورة"); return; }
        try (InputStream inputStream = getContentResolver().openInputStream(uri)) {
            Bitmap decoded = BitmapFactory.decodeStream(inputStream);
            if (decoded == null) { showMessage("تعذر قراءة الصورة"); return; }
            setCurrentBitmap(decoded, "تم تحميل الصورة: " + decoded.getWidth() + " x " + decoded.getHeight());
        } catch (IOException exception) { showMessage("فشل فتح الصورة"); }
    }

    private void loadVideoFrame(Uri uri) {
        if (uri == null) { showMessage("لم يتم العثور على مسار الفيديو"); return; }
        MediaMetadataRetriever retriever = new MediaMetadataRetriever();
        try {
            retriever.setDataSource(this, uri);
            Bitmap frame = retriever.getFrameAtTime(0, MediaMetadataRetriever.OPTION_CLOSEST_SYNC);
            if (frame == null) { showMessage("تعذر استخراج إطار من الفيديو"); return; }
            setCurrentBitmap(frame, "تم استخراج إطار من الفيديو: " + frame.getWidth() + " x " + frame.getHeight());
        } catch (RuntimeException exception) { showMessage("فشل قراءة الفيديو"); }
        finally { try { retriever.release(); } catch (IOException ignored) {} }
    }

    private void setCurrentBitmap(Bitmap bitmap, String status) {
        originalBitmap = bitmap.copy(Bitmap.Config.ARGB_8888, true);
        currentBitmap = originalBitmap.copy(Bitmap.Config.ARGB_8888, true);
        imageView.setImageBitmap(currentBitmap);
        statusText.setText(status);
    }

    private void applyFilter(FilterMode mode) {
        if (currentBitmap == null) { showMessage("اختر صورة أو فيديو أولًا"); return; }
        Bitmap source = currentBitmap.copy(Bitmap.Config.ARGB_8888, false);
        if (mode == FilterMode.GRAYSCALE) { currentBitmap = toGrayscale(source); statusText.setText("تم تطبيق Grayscale"); }
        else if (mode == FilterMode.THRESHOLD) { currentBitmap = toThreshold(source, 128); statusText.setText("تم تطبيق Threshold"); }
        else { currentBitmap = toEdges(source); statusText.setText("تم تطبيق Edges"); }
        imageView.setImageBitmap(currentBitmap);
    }

    private Bitmap toGrayscale(Bitmap source) {
        int width = source.getWidth(), height = source.getHeight();
        Bitmap output = Bitmap.createBitmap(width, height, Bitmap.Config.ARGB_8888);
        int[] pixels = new int[width * height]; source.getPixels(pixels, 0, width, 0, 0, width, height);
        for (int i = 0; i < pixels.length; i++) { int color = pixels[i]; int gray = luminance(color); pixels[i] = Color.argb(Color.alpha(color), gray, gray, gray); }
        output.setPixels(pixels, 0, width, 0, 0, width, height); return output;
    }
    private Bitmap toThreshold(Bitmap source, int threshold) {
        int width = source.getWidth(), height = source.getHeight();
        Bitmap output = Bitmap.createBitmap(width, height, Bitmap.Config.ARGB_8888);
        int[] pixels = new int[width * height]; source.getPixels(pixels, 0, width, 0, 0, width, height);
        for (int i = 0; i < pixels.length; i++) { int value = luminance(pixels[i]) >= threshold ? 255 : 0; pixels[i] = Color.argb(Color.alpha(pixels[i]), value, value, value); }
        output.setPixels(pixels, 0, width, 0, 0, width, height); return output;
    }
    private Bitmap toEdges(Bitmap source) {
        int width = source.getWidth(), height = source.getHeight();
        Bitmap output = Bitmap.createBitmap(width, height, Bitmap.Config.ARGB_8888);
        int[] pixels = new int[width * height]; int[] result = new int[width * height]; source.getPixels(pixels, 0, width, 0, 0, width, height);
        for (int y = 1; y < height - 1; y++) for (int x = 1; x < width - 1; x++) { int index = y * width + x; int magnitude = Math.min(255, Math.abs(luminance(pixels[index + 1]) - luminance(pixels[index - 1])) + Math.abs(luminance(pixels[index + width]) - luminance(pixels[index - width]))); result[index] = Color.rgb(magnitude, magnitude, magnitude); }
        output.setPixels(result, 0, width, 0, 0, width, height); return output;
    }
    private int luminance(int color) { return (int) (0.299f * Color.red(color) + 0.587f * Color.green(color) + 0.114f * Color.blue(color)); }
    private void restoreOriginal() { if (originalBitmap == null) { showMessage("اختر صورة أو فيديو أولًا"); return; } currentBitmap = originalBitmap.copy(Bitmap.Config.ARGB_8888, true); imageView.setImageBitmap(currentBitmap); statusText.setText("تمت استعادة الأصل"); }
    private void saveCurrentImage() {
        if (currentBitmap == null) { showMessage("لا توجد نتيجة لحفظها"); return; }
        ContentValues values = new ContentValues(); values.put(MediaStore.Images.Media.DISPLAY_NAME, "visionstudio_" + System.currentTimeMillis() + ".png"); values.put(MediaStore.Images.Media.MIME_TYPE, "image/png"); values.put(MediaStore.Images.Media.RELATIVE_PATH, "Pictures/VisionStudio");
        Uri uri = getContentResolver().insert(MediaStore.Images.Media.EXTERNAL_CONTENT_URI, values); if (uri == null) { showMessage("تعذر إنشاء ملف الحفظ"); return; }
        try (OutputStream outputStream = getContentResolver().openOutputStream(uri)) { currentBitmap.compress(Bitmap.CompressFormat.PNG, 100, outputStream); showMessage("تم حفظ النتيجة في Pictures/VisionStudio"); } catch (IOException exception) { showMessage("فشل حفظ الصورة"); }
    }
    private void showMessage(String message) { Toast.makeText(this, message, Toast.LENGTH_SHORT).show(); }
    private enum FilterMode { GRAYSCALE, THRESHOLD, EDGES }
}
''')
print('Android 0.2.0 project written to:', APP_DIR)


Android 0.2.0 project written to: /content/drive/MyDrive/VisionStudio_Android_Build/VisionStudioAndroid


## 4. تجهيز Gradle Wrapper


In [4]:
%cd $APP_DIR
!if command -v gradle >/dev/null 2>&1; then gradle wrapper --gradle-version 8.9; else cd /content && wget -q https://services.gradle.org/distributions/gradle-8.9-bin.zip -O gradle-8.9-bin.zip && unzip -q -o gradle-8.9-bin.zip && cd $APP_DIR && /content/gradle-8.9/bin/gradle wrapper --gradle-version 8.9; fi


/content/drive/MyDrive/VisionStudio_Android_Build/VisionStudioAndroid
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Welcome to Gradle 8.9!

Here are the highlights of this release:
 - Enhanced Error and Warning Messages
 - IDE Integration Improvements
 - Daemon JVM Information

For more details see https://docs.gradle.org/8.9/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)


> Starting Daemon> IDLE<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<----------

## 5. بناء APK

قد تستغرق هذه الخلية من 2 إلى 10 دقائق في أول تشغيل.


In [5]:
%cd $APP_DIR
!chmod +x ./gradlew
!./gradlew assembleDebug --stacktrace


/content/drive/MyDrive/VisionStudio_Android_Build/VisionStudioAndroid
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
............10%.............20%.............30%.............40%.............50%.............60%.............70%.............80%.............90%.............100%

<-------------> 0% INITIALIZING s]
<-------------> 0% CONFIGURING s]> Loading projects<-------------> 0% CONFIGURING s]> root project > Resolve dependencies of :classpath<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]> root project > Resolve files of configuration ':classpath'<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<----------

## 6. تحميل APK


In [6]:
from pathlib import Path

apk_path = APP_DIR / 'app/build/outputs/apk/debug/app-debug.apk'
print('APK exists:', apk_path.exists())
print('APK path:', apk_path)
if apk_path.exists():
    print('APK size MB:', round(apk_path.stat().st_size / (1024 * 1024), 2))
    from google.colab import files
    files.download(str(apk_path))


APK exists: True
APK path: /content/drive/MyDrive/VisionStudio_Android_Build/VisionStudioAndroid/app/build/outputs/apk/debug/app-debug.apk
APK size MB: 0.01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. اختبار سريع على الهاتف

بعد تثبيت APK جرّب:

1. اختيار صورة من المعرض.
2. التقاط صورة بالكاميرا.
3. اختيار فيديو واستخراج إطار.
4. تطبيق الفلاتر الثلاثة.
5. حفظ النتيجة.

إذا نجحت هذه النقاط، تكون نسخة الصور + الكاميرا + الفيديو الأساسي جاهزة للتسليم السريع.


## ملاحظة الدمج بين 11 و12

- وظائف الدفتر 11 الأساسية (اختيار صورة، Grayscale، Threshold، Edges، الحفظ) موجودة في الإصدار الأحدث.
- الدفتر 12 يضيف مسار الكاميرا والفيديو ويستخدم نفس `WORK_DIR` و`APP_DIR` ومسار APK.
- لذلك تم اعتماد كود Android الأحدث فقط لمنع استبدال ملفات الإصدار 0.2.0 بإصدار 0.1.0 أثناء التشغيل.
